---
title: Gamma Beta Network
project:
  type: website
format:
  html:
    code-fold: false
    code-tools: true
jupyter: python 3
number-sections: false
---

The goal in this notebook is to combine our previous [PING]() model and [beta rhythm]() model.

We'll combine these models to study how networks can synchronize their activity. 

# Connect two PING networks

To begin, let's combine two PING models.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def alphaM(V):
    return (2.5-0.1*(V+65)) / (np.exp(2.5-0.1*(V+65)) -1)

def betaM(V):
    return 4*np.exp(-(V+65)/18)

def alphaH(V):
    return 0.07*np.exp(-(V+65)/20)

def betaH(V):
    return 1/(np.exp(3.0-0.1*(V+65))+1)

def alphaN(V):
    return (0.1-0.01*(V+65)) / (np.exp(1-0.1*(V+65)) -1)

def betaN(V):
    return 0.125*np.exp(-(V+65)/80)

def alphaI(V):
    return ((1 + np.tanh(V / 10)))

def betaI(tauI):
    return 1/tauI

def alphaP(V):
    return 2.5*((1 + np.tanh(V / 10)))

def betaP(tauP):
    return 1/tauP

def delayed_voltage(V, i, delay_steps):
    j = i - delay_steps
    return V[j] if j >= 0 else V[0]

def two_ping(I0P, T0, gPtoI_cross=0, gPtoP_cross=0):
    
    dt = 0.01
    T  = int(np.ceil(T0 / dt))

    # Fix parameters
    gI   = 1
    gP   = 10
    tauI = 10
    tauP = 2
    I0I  = 0
    
    gNa0 = 120   # [mS/cm^2]
    ENa = 125    # [mV]
    gK0 = 36     # [mS/cm^2]
    EK = -12     # [mV]
    gL0 = 0.3    # [mS/cm^2]
    EL = 10.6    # [mV]

    t = np.arange(0, T) * dt
    VP1 = np.zeros(T); mP1 = np.zeros(T); hP1 = np.zeros(T); nP1 = np.zeros(T)
    VI1 = np.zeros(T); mI1 = np.zeros(T); hI1 = np.zeros(T); nI1 = np.zeros(T)
    VP2 = np.zeros(T); mP2 = np.zeros(T); hP2 = np.zeros(T); nP2 = np.zeros(T)
    VI2 = np.zeros(T); mI2 = np.zeros(T); hI2 = np.zeros(T); nI2 = np.zeros(T)

    sP1   = np.zeros(T); sI1 = np.zeros(T)
    sP2   = np.zeros(T); sI2 = np.zeros(T)
    sP1I2 = np.zeros(T); sP2I1 = np.zeros(T)
    sP1P2 = np.zeros(T); sP2P1 = np.zeros(T)

    for V, m, h, n in [(VP1, mP1, hP1, nP1), (VI1, mI1, hI1, nI1),
                       (VP2, mP2, hP2, nP2), (VI2, mI2, hI2, nI2)]:
        V[0] = -70.0 + np.random.randn()
        m[0] = 0.05  + 0.01*np.random.rand()
        h[0] = 0.54  + 0.2*np.random.rand()
        n[0] = 0.34  + 0.2*np.random.rand()

    for i in range(0, T - 1):
        VP1[i+1] = VP1[i] + dt*(
            gNa0*mP1[i]**3*hP1[i]*(ENa-(VP1[i]+65)) + gK0*nP1[i]**4*(EK-(VP1[i]+65)) + gL0*(EL-(VP1[i]+65))
            + I0P[0]
            + gI*sI1[i]*(-80 - VP1[i])
            + gPtoP_cross*sP2P1[i]*(0 - VP1[i]))
        mP1[i+1] = mP1[i] + dt*(alphaM(VP1[i])*(1-mP1[i]) - betaM(VP1[i])*mP1[i])
        hP1[i+1] = hP1[i] + dt*(alphaH(VP1[i])*(1-hP1[i]) - betaH(VP1[i])*hP1[i])
        nP1[i+1] = nP1[i] + dt*(alphaN(VP1[i])*(1-nP1[i]) - betaN(VP1[i])*nP1[i])

        VI1[i+1] = VI1[i] + dt*(
            gNa0*mI1[i]**3*hI1[i]*(ENa-(VI1[i]+65)) + gK0*nI1[i]**4*(EK-(VI1[i]+65)) + gL0*(EL-(VI1[i]+65))
            + I0I
            + gP*sP1[i]*(0 - VI1[i])
            + gPtoI_cross*sP2I1[i]*(0 - VI1[i]))
        mI1[i+1] = mI1[i] + dt*(alphaM(VI1[i])*(1-mI1[i]) - betaM(VI1[i])*mI1[i])
        hI1[i+1] = hI1[i] + dt*(alphaH(VI1[i])*(1-hI1[i]) - betaH(VI1[i])*hI1[i])
        nI1[i+1] = nI1[i] + dt*(alphaN(VI1[i])*(1-nI1[i]) - betaN(VI1[i])*nI1[i])

        VP2[i+1] = VP2[i] + dt*(
            gNa0*mP2[i]**3*hP2[i]*(ENa-(VP2[i]+65)) + gK0*nP2[i]**4*(EK-(VP2[i]+65)) + gL0*(EL-(VP2[i]+65))
            + I0P[1]
            + gI*sI2[i]*(-80 - VP2[i])
            + gPtoP_cross*sP1P2[i]*(0 - VP2[i]))
        mP2[i+1] = mP2[i] + dt*(alphaM(VP2[i])*(1-mP2[i]) - betaM(VP2[i])*mP2[i])
        hP2[i+1] = hP2[i] + dt*(alphaH(VP2[i])*(1-hP2[i]) - betaH(VP2[i])*hP2[i])
        nP2[i+1] = nP2[i] + dt*(alphaN(VP2[i])*(1-nP2[i]) - betaN(VP2[i])*nP2[i])

        VI2[i+1] = VI2[i] + dt*(
            gNa0*mI2[i]**3*hI2[i]*(ENa-(VI2[i]+65))
            + gK0*nI2[i]**4*(EK-(VI2[i]+65))
            + gL0*(EL-(VI2[i]+65))
            + I0I
            + gP*sP2[i]*(0 - VI2[i])
            + gPtoI_cross*sP1I2[i]*(0 - VI2[i]))
        mI2[i+1] = mI2[i] + dt*(alphaM(VI2[i])*(1-mI2[i]) - betaM(VI2[i])*mI2[i])
        hI2[i+1] = hI2[i] + dt*(alphaH(VI2[i])*(1-hI2[i]) - betaH(VI2[i])*hI2[i])
        nI2[i+1] = nI2[i] + dt*(alphaN(VI2[i])*(1-nI2[i]) - betaN(VI2[i])*nI2[i])

        sP1[i+1] = sP1[i] + dt*(alphaP(VP1[i])*(1-sP1[i]) - betaP(tauP)*sP1[i])
        sI1[i+1] = sI1[i] + dt*(alphaI(VI1[i])*(1-sI1[i]) - betaI(tauI)*sI1[i])
        sP2[i+1] = sP2[i] + dt*(alphaP(VP2[i])*(1-sP2[i]) - betaP(tauP)*sP2[i])
        sI2[i+1] = sI2[i] + dt*(alphaI(VI2[i])*(1-sI2[i]) - betaI(tauI)*sI2[i])

        sP1I2[i+1] = sP1I2[i] + dt*(alphaP(VP1[i])*(1-sP1I2[i]) - betaP(tauP)*sP1I2[i])
        sP2I1[i+1] = sP2I1[i] + dt*(alphaP(VP2[i])*(1-sP2I1[i]) - betaP(tauP)*sP2I1[i])
        sP1P2[i+1] = sP1P2[i] + dt*(alphaP(VP1[i])*(1-sP1P2[i]) - betaP(tauP)*sP1P2[i])
        sP2P1[i+1] = sP2P1[i] + dt*(alphaP(VP2[i])*(1-sP2P1[i]) - betaP(tauP)*sP2P1[i])

    return VP1, VI1, VP2, VI2, t

In [ ]:
# Fix parametes & Simulate
I0P         = [12, 10]      # More excitatory drive to first PING model.
gPtoP_cross = 0.0           # 0 or 0.1, P-to-P across networks
gPtoI_cross = 0.0           # 0 or 10,       P-to-I across networks
T0          = 250

[VP1, VI1, VP2, VI2, t] = two_ping( I0P, T0, gPtoI_cross=gPtoI_cross, gPtoP_cross=gPtoP_cross)

plt.figure(figsize=(10, 7))
plt.subplot(3, 1, 1)
plt.plot(t, VP1, label='VP1')
plt.plot(t, VP2, label='VP2')
plt.xlabel('Time [ms]'); plt.ylabel('P voltage [mV]'); plt.legend()

plt.subplot(3, 1, 2)
plt.plot(t, VI1, label='VI1')
plt.plot(t, VI2, label='VI2')
plt.xlabel('Time [ms]'); plt.ylabel('I voltage [mV]'); plt.legend()

# Two PING populations with delays

Update the model to include delays in the long-distance synaptic interactions.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def alphaM(V):
    return (2.5-0.1*(V+65)) / (np.exp(2.5-0.1*(V+65)) -1)

def betaM(V):
    return 4*np.exp(-(V+65)/18)

def alphaH(V):
    return 0.07*np.exp(-(V+65)/20)

def betaH(V):
    return 1/(np.exp(3.0-0.1*(V+65))+1)

def alphaN(V):
    return (0.1-0.01*(V+65)) / (np.exp(1-0.1*(V+65)) -1)

def betaN(V):
    return 0.125*np.exp(-(V+65)/80)

def alphaI(V):
    return ((1 + np.tanh(V / 10)))

def betaI(tauI):
    return 1/tauI

def alphaP(V):
    return 2.5*((1 + np.tanh(V / 10)))

def betaP(tauP):
    return 1/tauP

def two_ping_with_delay(I0P, T0, gPtoI_cross=0, gPtoP_cross=0, delay_cross=0):
    
    dt = 0.01
    T  = int(np.ceil(T0 / dt))

    delay_steps = int(np.round(delay_cross / dt))
    if delay_steps < 0:
        raise ValueError('Synaptic delays must be nonnegative.')

    # Fix parameters
    gI   = 1
    gP   = 10
    tauI = 10
    tauP = 2
    I0I  = 0
    
    gNa0 = 120   # [mS/cm^2]
    ENa = 125    # [mV]
    gK0 = 36     # [mS/cm^2]
    EK = -12     # [mV]
    gL0 = 0.3    # [mS/cm^2]
    EL = 10.6    # [mV]

    t = np.arange(0, T) * dt

    VP1 = np.zeros(T); mP1 = np.zeros(T); hP1 = np.zeros(T); nP1 = np.zeros(T)
    VI1 = np.zeros(T); mI1 = np.zeros(T); hI1 = np.zeros(T); nI1 = np.zeros(T)
    VP2 = np.zeros(T); mP2 = np.zeros(T); hP2 = np.zeros(T); nP2 = np.zeros(T)
    VI2 = np.zeros(T); mI2 = np.zeros(T); hI2 = np.zeros(T); nI2 = np.zeros(T)

    sP1   = np.zeros(T); sI1 = np.zeros(T)
    sP2   = np.zeros(T); sI2 = np.zeros(T)
    sP1I2 = np.zeros(T); sP2I1 = np.zeros(T)
    sP1P2 = np.zeros(T); sP2P1 = np.zeros(T)

    for V, m, h, n in [(VP1, mP1, hP1, nP1), (VI1, mI1, hI1, nI1),
                       (VP2, mP2, hP2, nP2), (VI2, mI2, hI2, nI2)]:
        V[0] = -70.0 + np.random.randn()
        m[0] = 0.05  + 0.01*np.random.rand()
        h[0] = 0.54  + 0.2*np.random.rand()
        n[0] = 0.34  + 0.2*np.random.rand()

    def delayed_voltage(V, i, delay_steps):
        j = i - delay_steps
        return V[j] if j >= 0 else V[0]
        
    for i in range(0, T - 1):
        VP1[i+1] = VP1[i] + dt*(
            gNa0*mP1[i]**3*hP1[i]*(ENa-(VP1[i]+65)) + gK0*nP1[i]**4*(EK-(VP1[i]+65)) + gL0*(EL-(VP1[i]+65))
            + I0P[0]
            + gI*sI1[i]*(-80 - VP1[i])
            + gPtoP_cross*sP2P1[i]*(0 - VP1[i]))
        mP1[i+1] = mP1[i] + dt*(alphaM(VP1[i])*(1-mP1[i]) - betaM(VP1[i])*mP1[i])
        hP1[i+1] = hP1[i] + dt*(alphaH(VP1[i])*(1-hP1[i]) - betaH(VP1[i])*hP1[i])
        nP1[i+1] = nP1[i] + dt*(alphaN(VP1[i])*(1-nP1[i]) - betaN(VP1[i])*nP1[i])

        VI1[i+1] = VI1[i] + dt*(
            gNa0*mI1[i]**3*hI1[i]*(ENa-(VI1[i]+65)) + gK0*nI1[i]**4*(EK-(VI1[i]+65)) + gL0*(EL-(VI1[i]+65))
            + I0I
            + gP*sP1[i]*(0 - VI1[i])
            + gPtoI_cross*sP2I1[i]*(0 - VI1[i]))
        mI1[i+1] = mI1[i] + dt*(alphaM(VI1[i])*(1-mI1[i]) - betaM(VI1[i])*mI1[i])
        hI1[i+1] = hI1[i] + dt*(alphaH(VI1[i])*(1-hI1[i]) - betaH(VI1[i])*hI1[i])
        nI1[i+1] = nI1[i] + dt*(alphaN(VI1[i])*(1-nI1[i]) - betaN(VI1[i])*nI1[i])

        VP2[i+1] = VP2[i] + dt*(
            gNa0*mP2[i]**3*hP2[i]*(ENa-(VP2[i]+65)) + gK0*nP2[i]**4*(EK-(VP2[i]+65)) + gL0*(EL-(VP2[i]+65))
            + I0P[1]
            + gI*sI2[i]*(-80 - VP2[i])
            + gPtoP_cross*sP1P2[i]*(0 - VP2[i]))
        mP2[i+1] = mP2[i] + dt*(alphaM(VP2[i])*(1-mP2[i]) - betaM(VP2[i])*mP2[i])
        hP2[i+1] = hP2[i] + dt*(alphaH(VP2[i])*(1-hP2[i]) - betaH(VP2[i])*hP2[i])
        nP2[i+1] = nP2[i] + dt*(alphaN(VP2[i])*(1-nP2[i]) - betaN(VP2[i])*nP2[i])

        VI2[i+1] = VI2[i] + dt*(
            gNa0*mI2[i]**3*hI2[i]*(ENa-(VI2[i]+65))
            + gK0*nI2[i]**4*(EK-(VI2[i]+65))
            + gL0*(EL-(VI2[i]+65))
            + I0I
            + gP*sP2[i]*(0 - VI2[i])
            + gPtoI_cross*sP1I2[i]*(0 - VI2[i]))
        mI2[i+1] = mI2[i] + dt*(alphaM(VI2[i])*(1-mI2[i]) - betaM(VI2[i])*mI2[i])
        hI2[i+1] = hI2[i] + dt*(alphaH(VI2[i])*(1-hI2[i]) - betaH(VI2[i])*hI2[i])
        nI2[i+1] = nI2[i] + dt*(alphaN(VI2[i])*(1-nI2[i]) - betaN(VI2[i])*nI2[i])

        # Short distance synapses
        sP1[i+1] = sP1[i] + dt*(alphaP(VP1[i])*(1-sP1[i]) - betaP(tauP)*sP1[i])
        sI1[i+1] = sI1[i] + dt*(alphaI(VI1[i])*(1-sI1[i]) - betaI(tauI)*sI1[i])
        sP2[i+1] = sP2[i] + dt*(alphaP(VP2[i])*(1-sP2[i]) - betaP(tauP)*sP2[i])
        sI2[i+1] = sI2[i] + dt*(alphaI(VI2[i])*(1-sI2[i]) - betaI(tauI)*sI2[i])

        # Long distance synapses
        sP1I2[i+1] = sP1I2[i] + dt*(alphaP(delayed_voltage(VP1, i, delay_steps))*(1-sP1I2[i]) - betaP(tauP)*sP1I2[i])
        sP2I1[i+1] = sP2I1[i] + dt*(alphaP(delayed_voltage(VP2, i, delay_steps))*(1-sP2I1[i]) - betaP(tauP)*sP2I1[i])
        sP1P2[i+1] = sP1P2[i] + dt*(alphaP(delayed_voltage(VP1, i, delay_steps))*(1-sP1P2[i]) - betaP(tauP)*sP1P2[i])
        sP2P1[i+1] = sP2P1[i] + dt*(alphaP(delayed_voltage(VP2, i, delay_steps))*(1-sP2P1[i]) - betaP(tauP)*sP2P1[i])

    return VP1, VI1, VP2, VI2, t

In [ ]:
# Fix parameters
I0P         = [12, 10]      # More excitatory drive to first PING model.
gPtoP_cross = 0.1           # 0.1, P-to-P across networks
gPtoI_cross = 10            # 10,  P-to-I across networks
delay_cross = 0             # 0 or 10 ms delay
T0          = 500

# Simulate
[VP1, VI1, VP2, VI2, t] = two_ping_with_delay( I0P, T0, gPtoI_cross=gPtoI_cross, gPtoP_cross=gPtoP_cross, delay_cross=delay_cross)

plt.figure(figsize=(10, 7))
plt.subplot(3, 1, 1)
plt.plot(t, VP1, label='VP1')
plt.plot(t, VP2, label='VP2')
plt.xlabel('Time [ms]'); plt.ylabel('P voltage [mV]'); plt.legend()

plt.subplot(3, 1, 2)
plt.plot(t, VI1, label='VI1')
plt.plot(t, VI2, label='VI2')
plt.xlabel('Time [ms]'); plt.ylabel('I voltage [mV]'); plt.legend()

# Gamma + beta model

Include a slow M-current in the pyramidal cells of each model.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def alphaM(V):
    return (2.5-0.1*(V+65)) / (np.exp(2.5-0.1*(V+65)) -1)

def betaM(V):
    return 4*np.exp(-(V+65)/18)

def alphaH(V):
    return 0.07*np.exp(-(V+65)/20)

def betaH(V):
    return 1/(np.exp(3.0-0.1*(V+65))+1)

def alphaN(V):
    return (0.1-0.01*(V+65)) / (np.exp(1-0.1*(V+65)) -1)

def betaN(V):
    return 0.125*np.exp(-(V+65)/80)

def alphaI(V):
    return ((1 + np.tanh(V / 10)))

def betaI(tauI):
    return 1/tauI

def alphaP(V):
    return 2.5*((1 + np.tanh(V / 10)))

def betaP(tauP):
    return 1/tauP

def alphaB(V):
    return 0.02 / (1+ np.exp((-V-20)/5))

def betaB(V):
    return 0.01 * np.exp((-V-43)/18)

def ping_beta(I0P, gB0, T0, gPtoI_cross=1, gPtoP_cross=0, delay_cross=0):
    
    dt = 0.01
    T = int(np.ceil(T0 / dt))  # [ms]
    delay_steps = int(np.round(delay_cross / dt))
    if delay_steps < 0:
        raise ValueError('Synaptic delays must be nonnegative.')

    # Fix parameters
    gI   = 1
    gP   = 10
    tauI = 10
    tauP = 2
    I0I  = 0

    gNa0 = 120   # [mS/cm^2]
    ENa = 125    # [mV]
    gK0 = 36     # [mS/cm^2]
    EK = -12     # [mV]
    gL0 = 0.3    # [mS/cm^2]
    EL = 10.6    # [mV]

    t = np.arange(0, T) * dt

    VP1 = np.zeros(T); mP1 = np.zeros(T); hP1 = np.zeros(T); nP1 = np.zeros(T); BP1 = np.zeros(T)
    VI1 = np.zeros(T); mI1 = np.zeros(T); hI1 = np.zeros(T); nI1 = np.zeros(T)
    VP2 = np.zeros(T); mP2 = np.zeros(T); hP2 = np.zeros(T); nP2 = np.zeros(T); BP2 = np.zeros(T)
    VI2 = np.zeros(T); mI2 = np.zeros(T); hI2 = np.zeros(T); nI2 = np.zeros(T)

    sP1 = np.zeros(T); sI1 = np.zeros(T)
    sP2 = np.zeros(T); sI2 = np.zeros(T)
    sP1I2 = np.zeros(T); sP2I1 = np.zeros(T)
    sP1P2 = np.zeros(T); sP2P1 = np.zeros(T)

    for V, m, h, n in [(VP1, mP1, hP1, nP1), (VI1, mI1, hI1, nI1),
                       (VP2, mP2, hP2, nP2), (VI2, mI2, hI2, nI2)]:
        V[0] = -70.0 + np.random.randn()
        m[0] = 0.05  + 0.01*np.random.rand()
        h[0] = 0.54  + 0.2*np.random.rand()
        n[0] = 0.34  + 0.2*np.random.rand()

    def delayed_voltage(V, i, delay_steps):
        j = i - delay_steps
        return V[j] if j >= 0 else V[0]

    for i in range(0, T - 1):
        VP1[i+1] = VP1[i] + dt*(
            gNa0*mP1[i]**3*hP1[i]*(ENa-(VP1[i]+65)) + gK0*nP1[i]**4*(EK-(VP1[i]+65)) + gL0*(EL-(VP1[i]+65))
            + I0P[0]
            + gI*sI1[i]*(-80 - VP1[i])
            + gB0*BP1[i]*(EK-(VP1[i]+65))
            + gPtoP_cross*sP2P1[i]*(0 - VP1[i]))
        mP1[i+1] = mP1[i] + dt*(alphaM(VP1[i])*(1-mP1[i]) - betaM(VP1[i])*mP1[i])
        hP1[i+1] = hP1[i] + dt*(alphaH(VP1[i])*(1-hP1[i]) - betaH(VP1[i])*hP1[i])
        nP1[i+1] = nP1[i] + dt*(alphaN(VP1[i])*(1-nP1[i]) - betaN(VP1[i])*nP1[i])
        BP1[i+1] = BP1[i] + dt*(alphaB(VP1[i])*(1-BP1[i]) - betaB(VP1[i])*BP1[i])

        VI1[i+1] = VI1[i] + dt*(
            gNa0*mI1[i]**3*hI1[i]*(ENa-(VI1[i]+65)) + gK0*nI1[i]**4*(EK-(VI1[i]+65)) + gL0*(EL-(VI1[i]+65))
            + I0I
            + gP*sP1[i]*(0 - VI1[i])
            + gPtoI_cross*sP2I1[i]*(0 - VI1[i]))
        mI1[i+1] = mI1[i] + dt*(alphaM(VI1[i])*(1-mI1[i]) - betaM(VI1[i])*mI1[i])
        hI1[i+1] = hI1[i] + dt*(alphaH(VI1[i])*(1-hI1[i]) - betaH(VI1[i])*hI1[i])
        nI1[i+1] = nI1[i] + dt*(alphaN(VI1[i])*(1-nI1[i]) - betaN(VI1[i])*nI1[i])

        VP2[i+1] = VP2[i] + dt*(
            gNa0*mP2[i]**3*hP2[i]*(ENa-(VP2[i]+65))
            + gK0*nP2[i]**4*(EK-(VP2[i]+65))
            + gL0*(EL-(VP2[i]+65))
            + I0P[1]
            + gI*sI2[i]*(-80 - VP2[i])
            + gB0*BP2[i]*(EK-(VP2[i]+65))
            + gPtoP_cross*sP1P2[i]*(0 - VP2[i]))
        mP2[i+1] = mP2[i] + dt*(alphaM(VP2[i])*(1-mP2[i]) - betaM(VP2[i])*mP2[i])
        hP2[i+1] = hP2[i] + dt*(alphaH(VP2[i])*(1-hP2[i]) - betaH(VP2[i])*hP2[i])
        nP2[i+1] = nP2[i] + dt*(alphaN(VP2[i])*(1-nP2[i]) - betaN(VP2[i])*nP2[i])
        BP2[i+1] = BP2[i] + dt*(alphaB(VP2[i])*(1-BP2[i]) - betaB(VP2[i])*BP2[i])

        VI2[i+1] = VI2[i] + dt*(
            gNa0*mI2[i]**3*hI2[i]*(ENa-(VI2[i]+65))
            + gK0*nI2[i]**4*(EK-(VI2[i]+65))
            + gL0*(EL-(VI2[i]+65))
            + I0I
            + gP*sP2[i]*(0 - VI2[i])
            + gPtoI_cross*sP1I2[i]*(0 - VI2[i]))
        mI2[i+1] = mI2[i] + dt*(alphaM(VI2[i])*(1-mI2[i]) - betaM(VI2[i])*mI2[i])
        hI2[i+1] = hI2[i] + dt*(alphaH(VI2[i])*(1-hI2[i]) - betaH(VI2[i])*hI2[i])
        nI2[i+1] = nI2[i] + dt*(alphaN(VI2[i])*(1-nI2[i]) - betaN(VI2[i])*nI2[i])

        sP1[i+1] = sP1[i] + dt*(alphaP(VP1[i])*(1-sP1[i]) - betaP(tauP)*sP1[i])
        sI1[i+1] = sI1[i] + dt*(alphaI(VI1[i])*(1-sI1[i]) - betaI(tauI)*sI1[i])
        sP2[i+1] = sP2[i] + dt*(alphaP(VP2[i])*(1-sP2[i]) - betaP(tauP)*sP2[i])
        sI2[i+1] = sI2[i] + dt*(alphaI(VI2[i])*(1-sI2[i]) - betaI(tauI)*sI2[i])

        sP1I2[i+1] = sP1I2[i] + dt*(alphaP(delayed_voltage(VP1, i, delay_steps))*(1-sP1I2[i]) - betaP(tauP)*sP1I2[i])
        sP2I1[i+1] = sP2I1[i] + dt*(alphaP(delayed_voltage(VP2, i, delay_steps))*(1-sP2I1[i]) - betaP(tauP)*sP2I1[i])
        sP1P2[i+1] = sP1P2[i] + dt*(alphaP(delayed_voltage(VP1, i, delay_steps))*(1-sP1P2[i]) - betaP(tauP)*sP1P2[i])
        sP2P1[i+1] = sP2P1[i] + dt*(alphaP(delayed_voltage(VP2, i, delay_steps))*(1-sP2P1[i]) - betaP(tauP)*sP2P1[i])

    return VP1, VI1, VP2, VI2, t

In [ ]:
# Change these values to control the cross-pair delays and P-to-P coupling strength.

np.random.seed(4)

# Fix parameters
I0P         = [12, 10]      # More excitatory drive to first PING model.
gPtoP_cross = 0.1           # Distant P-to-P synapses
gPtoI_cross = 10            # Distant P-to-I synapses 
delay_cross = 10            # 0 or 10 ms, delay between populations
gB          = 0             # 0 or 4, M-current max conductance
T0          = 500

# Simulate
[VP1, VI1, VP2, VI2, t] = ping_beta(I0P, gB, T0,
    gPtoI_cross=gPtoI_cross,
    delay_cross=delay_cross,
    gPtoP_cross=gPtoP_cross)

plt.figure(figsize=(10, 7))
plt.subplot(3, 1, 1)
plt.plot(t, VP1, label='VP1')
plt.plot(t, VP2, label='VP2')
plt.xlabel('Time [ms]')
plt.ylabel('P voltage [mV]')
plt.legend()

plt.subplot(3, 1, 2)
plt.plot(t, VI1, label='VI1')
plt.plot(t, VI2, label='VI2')
plt.xlabel('Time [ms]')
plt.ylabel('I voltage [mV]')
plt.legend()